# V4.1 — Residual-Faithful Global + CeNN Distillation (Group 18–20)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V41_ResidualFaithful_CeNN_Group18_20_Colab.ipynb)

**Purpose:** investigate why V4.0 training hurt decisions even though group latency improved. This is a new experiment; V4.0 checkpoints must not be reused.

**Evidence from V4.0 AUDIT (200 decisions):** teacher agreement 80.5%, gold accuracy 73.5%; disabling local CeNN updates yields 90.0%, 80.0%, respectively. Student group was 2.04× faster, but not sufficiently faithful.

### Mandatory wiring: two original sliding blocks each have TWO residual branches
```
Original GLOBAL layer 18 (frozen pretrained SDPA + FFN)
  → h18
  → block 19: a19 = h18 + Attention19(LN_att19(h18))
              h19 = a19 + MLP19(LN_mlp19(a19))
  → block 20: a20 = h19 + Attention20(LN_att20(h19))
              h20 = a20 + MLP20(LN_mlp20(a20))

Replacement (same residual identities and order):
  → h18 (original native global, W1024 = unrestricted for T<=512)
  → CeNN step19: a19' = h18 + gA19 · CeNN_shared(LN_att19(h18), valid)
                   h19' = a19' + gF19 · LowRankFFN19(LN_mlp19(a19'))
  → CeNN step20: a20' = h19' + gA20 · CeNN_shared(LN_att20(h19'), valid)
                   h20' = a20' + gF20 · LowRankFFN20(LN_mlp20(a20'))
```
The two local **spatial** CeNN iterations share convolution and channel weights, but use *separate frozen pretrained layer norms* and *separate low-rank FFNs* for the two sliding steps. The optional long-range RF1025 branch is independent and enters the **attention-like residual**, never replacing the identity stream. There is no extra standalone residual.

**Learning strategy:** cache exact original **a19/h19/a20/h20** teacher activations, supervise each of the four stages (not just y19/y20), train from the correct h18 prefix, verify zero-update identity, gradient nonzero, weight freezing and forward parity; then optionally perform student-prefix, task-level KL + gold recovery with PROBE-only checkpoint choice and rollback to the no-update baseline when needed. Evaluate AUDIT once only after selection. One pretrained model on GPU.

**Caution:** this is a faithful residual *topology*, not a mathematically exact replacement of pretrained SDPA and pretrained FFNs. V4.0 low cosine despite low aggregate NMSE suggests outlier/near-zero token effects too; V4.1 logs stage-specific cosine, NMSE and gradient norms. No success is claimed before GPU execution.

**New output folder:** /content/Laya_V41_ResidualFaithful, deliberately separate from V4.0. Train only group 18–20, prove no-regression before attempting other 3-to-1 groups.

### FP16 instability correction
An executed V4.1 T4 run stopped after 200 local updates with a non-finite loss. Earlier diagnostics showed a19 rRMSE≈0.016, whereas the full h19 output remained at ≈0.80 rRMSE: the near-zero-initialized low-rank FFN, rather than only the CeNN attention wiring, is the main bottleneck. **This updated notebook trains the small adapter in FP32, lowers LR to 2×10⁻⁴, clips gradients, rolls back to last finite parameters after a bad batch, and compares an independent original-pretrained-FFN diagnostic bridge.** Original native global attention and FFNs remain frozen. A checkpoint is selected only on PROBE and the default no-update control is retained.


In [ ]:
#@title 1. Install pinned V4.1 dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Frozen pretrained Laya and V4.1 experiment configuration
from pathlib import Path
from contextlib import contextmanager
import copy, json, math, random, time, hashlib, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question, full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2041
TARGET=(18,19,20)
MAX_LEN=512 #@param {type:"integer"}
ALLOW_EXTENDED_CONTEXT=False #@param {type:"boolean"}
FULL_WINDOWS=(1024,)
LOCAL_RADIUS=128
LONG_RF=1025
CENN_RANK=96 #@param {type:"integer"}
# The V4.0 winning full W1024 is left original while wiring is corrected.
USE_WIDE_BRANCH=False #@param {type:"boolean"}
TIE_LOCAL_WEIGHTS=True #@param {type:"boolean"}
FFN_RANK=128 #@param {type:"integer"}
LOCAL_TRAIN_STEPS=360 #@param {type:"integer"}
TASK_CALIBRATION_STEPS=100 #@param {type:"integer"}
RUN_TASK_CALIBRATION=True #@param {type:"boolean"}
TASK_TRAIN_BATCHES=48 #@param {type:"integer"}
TASK_LEARNING_RATE=0.0001 #@param {type:"number"}
SUPERVISE_ATTENTION_INTERMEDIATES=True #@param {type:"boolean"}
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES=48 #@param {type:"integer"}
PROBE_CACHE_BATCHES=12 #@param {type:"integer"}
AUDIT_CACHE_BATCHES=12 #@param {type:"integer"}
SHORT_TRAIN_STEPS=100 #@param {type:"integer"}
REFINE_EXTRA_STEPS=300 #@param {type:"integer"}
TOP_DISTINCT_WINDOWS=1 #@param {type:"integer"}
REFINE_TOP_CANDIDATES=2 #@param {type:"integer"}
RUN_UNTIED_CONTROL=True #@param {type:"boolean"}
WINDOW_SCREEN_DECISIONS=100 #@param {type:"integer"}
CANDIDATE_PROBE_DECISIONS=100 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
CHECK_EVERY=50 #@param {type:"integer"}
LEARNING_RATE=0.0002 #@param {type:"number"}
TRAIN_FP32_ADAPTER=True #@param {type:"boolean"}
MAX_GRAD_NORM=0.25 #@param {type:"number"}
RECOVERY_LR_FACTOR=0.5 #@param {type:"number"}
MAX_RECOVERY_EVENTS=12 #@param {type:"integer"}
LOG_STAGE_ENERGY=True #@param {type:"boolean"}
BENCH_REPEATS=8 #@param {type:"integer"}
RESUME=True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
OUTPUT_DIR=Path("/content/Laya_V41_ResidualFaithful")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR=Path("/content/drive/MyDrive/TinyCeNN-LM/V41_ResidualFaithful")
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert TARGET==(18,19,20) and 1<=BATCH<=4
assert 1<=TOP_DISTINCT_WINDOWS<=len(FULL_WINDOWS)
assert 1<=REFINE_TOP_CANDIDATES<=12
assert MAX_LEN<=1024 and TRAIN_CACHE_BATCHES>=1 and PROBE_CACHE_BATCHES>=1
assert TRAIN_CASES+DEV_CASES<=1200
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
FULL_LAYERS=full_attention_indices(teacher)
assert list(FULL_LAYERS)==[0,3,6,9,12,15,18,21,24,27]
assert len(teacher.encoder.layers)==28
assert TARGET[0] in FULL_LAYERS and TARGET[1] not in FULL_LAYERS and TARGET[2] not in FULL_LAYERS
assert int(teacher.encoder.config.local_attention)//2==64
if ALLOW_EXTENDED_CONTEXT:
    limit=int(getattr(teacher.encoder.config,"max_position_embeddings",0))
    if MAX_LEN>limit:
        raise RuntimeError(f"Extended length {MAX_LEN} exceeds encoder position limit {limit}.")
elif MAX_LEN>int(source_cfg.get("max_len",MAX_LEN)):
    raise RuntimeError("Use MAX_LEN at or below source trained length, or explicitly opt into extended context.")
print("Pretrained original layers:",len(teacher.encoder.layers),"target",TARGET)
print("Native slide radius",int(teacher.encoder.config.local_attention)//2)
print("Full window candidates",FULL_WINDOWS,"local CeNN radius",LOCAL_RADIUS,
      "long-range RF",LONG_RF)
print("CUDA:",torch.cuda.get_device_name(0),"AMP:",amp_dtype)
print("Task pretrained max_len",source_cfg.get("max_len"),"requested",MAX_LEN)


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=(MAX_LEN if ALLOW_EXTENDED_CONTEXT else
                         min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN)))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.0 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Exact ModernBERT full-window modifier: preserve original RoPE, FFN and native SDPA
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_FULL=ORIGINAL_LAYERS[18]
@torch.no_grad()
def observed_window_equivalence(window,observed_max):
    """Keys in a centered total-width W mask have no excluded positions if W>=2*T-1,
    but at T<=W some queries near edges can still have remote keys. We therefore
    never infer equivalence from W>=T alone: use the actual mask comparison.
    """
    T=int(observed_max)
    if T==0:return False
    return torch.equal(make_band(T,window,torch.device("cpu")),
                       torch.ones((T,T),dtype=torch.bool))

def make_band(T,width,device):
    width=int(width)
    left=(width-1)//2
    right=width-1-left
    pos=torch.arange(T,device=device)
    delta=pos[None,:]-pos[:,None]
    return (delta>=-left)&(delta<=right)

class GlobalWindow18(nn.Module):
    """Wrap the FULL original block, modifying only its additive full mask.

    Original sliding_window_mask remains unchanged, and is passed through.
    For the configured full token window W, query i can attend key j iff
    -floor((W-1)/2) <= j-i <= ceil((W-1)/2).
    """
    def __init__(self,original,width):
        super().__init__()
        self.original=original
        self.width=int(width)
    def forward(self,hidden_states,*args,**kwargs):
        attention_mask=kwargs.get("attention_mask")
        T=hidden_states.shape[1]
        if attention_mask is None or attention_mask.ndim!=4:
            raise RuntimeError("Expected encoder-supplied 4D full additive mask.")
        if attention_mask.shape[-2:]!=(T,T):
            raise RuntimeError(f"Incorrect full mask shape {attention_mask.shape}")
        band=make_band(T,self.width,hidden_states.device)
        if attention_mask.dtype==torch.bool:
            modified=attention_mask & band[None,None,:,:]
        else:
            modified=attention_mask.masked_fill(
                ~band[None,None,:,:],torch.finfo(attention_mask.dtype).min)
        # Never change the native sliding mask in this window ablation.
        args_kwargs=dict(kwargs,attention_mask=modified)
        return self.original(hidden_states,*args,**args_kwargs)

@contextmanager
def native_window_only(width):
    assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i]
               for i in range(28)),"Encoder not restored before window screening."
    try:
        teacher.encoder.layers[18]=GlobalWindow18(ORIGINAL_FULL,width)
        yield teacher
    finally:
        teacher.encoder.layers[18]=ORIGINAL_FULL

@torch.inference_mode()
def teacher_probability_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        selected=items[start:min(start+BATCH,limit)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        p=torch.softmax(logits.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),-1).cpu()
        for row,item in zip(p,selected):
            out.append(dict(probs=row,teacher=int(row.argmax()),
                            gold=int(item["gold_index"]),
                            case_id=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def decision_score(items,reference):
    count=len(reference)
    agreements=correct=teacher_correct=0
    js=0.
    for start in range(0,count,BATCH):
        selected=items[start:min(start+BATCH,count)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        p=torch.softmax(logits.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),-1).cpu()
        for j,item in enumerate(selected):
            idx=start+j
            t=reference[idx]["probs"];s=p[j]
            assert t.shape==s.shape
            middle=(t+s)/2
            js+=float(.5*((t*(t.clamp_min(1e-8).log()-
                                middle.clamp_min(1e-8).log())).sum()+
                           (s*(s.clamp_min(1e-8).log()-
                                middle.clamp_min(1e-8).log())).sum()))
            predicted=int(s.argmax())
            agreements+=int(predicted==reference[idx]["teacher"])
            correct+=int(predicted==item["gold_index"])
            teacher_correct+=int(reference[idx]["teacher"]==item["gold_index"])
    return dict(decisions=count,teacher_agreement=agreements/count,
                teacher_gold_accuracy=teacher_correct/count,
                student_gold_accuracy=correct/count,
                gold_accuracy_drop=(teacher_correct-correct)/count,
                mean_js=js/count)

print("Native attention window wrapper configured.")


In [ ]:
#@title 5. Confirm W1024 native full-attention equivalence on PROBE
observed_lengths=[len(item["ids"]) for item in probe_items[:WINDOW_SCREEN_DECISIONS]]
assert observed_lengths,"No PROBE examples."
observed_max=max(observed_lengths)
print("Actual PROBE lengths:",min(observed_lengths),np.median(observed_lengths),
      observed_max)
window_identical={}
for W in FULL_WINDOWS:
    band=make_band(observed_max,W,torch.device("cpu"))
    ident=[prior for prior in window_identical
           if torch.equal(band,make_band(observed_max,prior,torch.device("cpu")))]
    window_identical[W]=(ident[0] if ident else W)
print("Equivalence classes at observed maximum T:",
      {str(w):int(v) for w,v in window_identical.items()})
if len(set(window_identical.values()))<len(FULL_WINDOWS):
    print("WARNING: These large full windows have IDENTICAL attention masks "
          "on the observed dataset. They are not independent comparisons.")
probe_teacher=teacher_probability_reference(probe_items,WINDOW_SCREEN_DECISIONS)
assert len(probe_teacher)==WINDOW_SCREEN_DECISIONS
baseline=decision_score(probe_items,probe_teacher)
assert baseline["teacher_agreement"]==1 and baseline["mean_js"]<1e-7
full_window_screen={}
for W in FULL_WINDOWS:
    reference_W=window_identical[W]
    if W!=reference_W:
        full_window_screen[str(W)]={
            **full_window_screen[str(reference_W)],
            "equivalent_to":reference_W,
            "independently_measured":False
        }
        print(f"Full W{W:4d} same mask as W{reference_W}; reuse metrics.")
        continue
    with native_window_only(W):
        score=decision_score(probe_items,probe_teacher)
    full_window_screen[str(W)]=dict(
        quality=score,equivalent_to=W,independently_measured=True)
    print(f"Full W{W:4d}: agreement={score['teacher_agreement']:.4f} "
          f"gold={score['student_gold_accuracy']:.4f}, JS={score['mean_js']:.6f}")

unique=list(dict.fromkeys(window_identical.values()))
unique.sort(key=lambda w:(
    -full_window_screen[str(w)]["quality"]["teacher_agreement"],
    full_window_screen[str(w)]["quality"]["gold_accuracy_drop"],
    full_window_screen[str(w)]["quality"]["mean_js"]
))
chosen_windows=unique[:TOP_DISTINCT_WINDOWS]
# Ensure the original unrestricted behavior is represented as a control.
full_window_native=next(
    (w for w in unique if observed_window_equivalence(w,observed_max)),None
)
if full_window_native is not None and full_window_native not in chosen_windows:
    chosen_windows[-1]=full_window_native
chosen_windows=list(dict.fromkeys(chosen_windows))
print("Selected distinct full windows for CeNN training:",chosen_windows)
print("Selection uses PROBE only, never AUDIT.")


In [ ]:
#@title 6. Capture exact original two-residual teacher stages: a19, h19, a20, h20
def cpu_native(t):
    # Preserve full residual-stream precision. CPU storage is used only for
    # a small, fixed set of teacher activation batches. No GPU teacher copy.
    return t.detach().to("cpu").contiguous()

def cuda_grad_safe(row):
    # Cache was captured under inference_mode, so clone outside that mode
    # before autograd can save activation tensors in nn.Linear backward.
    return {key:value.to(device,non_blocking=True).clone()
            for key,value in row.items()}

@torch.inference_mode()
def capture_residual_stages(items,n_batches,label):
    assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i] for i in range(28))
    result=[]
    for n in range(n_batches):
        selected=items[n*BATCH:(n+1)*BATCH]
        if len(selected)<BATCH:break
        b=direct_batch(selected)
        snap={}
        def input_hook(key):
            def fn(mod,args,kwargs):
                snap[key]=args[0].detach()
            return fn
        def output_hook(key):
            def fn(mod,args,kwargs,output):
                snap[key]=output[0].detach()
            return fn
        hooks=[
            ORIGINAL_LAYERS[18].register_forward_hook(
                output_hook("h18"),with_kwargs=True),
            ORIGINAL_LAYERS[19].register_forward_pre_hook(
                input_hook("x19"),with_kwargs=True),
            ORIGINAL_LAYERS[19].attn.register_forward_hook(
                output_hook("att19"),with_kwargs=True),
            ORIGINAL_LAYERS[19].register_forward_hook(
                output_hook("h19"),with_kwargs=True),
            ORIGINAL_LAYERS[20].register_forward_pre_hook(
                input_hook("x20"),with_kwargs=True),
            ORIGINAL_LAYERS[20].attn.register_forward_hook(
                output_hook("att20"),with_kwargs=True),
            ORIGINAL_LAYERS[20].register_forward_hook(
                output_hook("h20"),with_kwargs=True),
        ]
        try:
            with torch.autocast("cuda",dtype=amp_dtype):
                teacher(**b)
        finally:
            for handle in hooks:handle.remove()
        assert set(snap)=={
            "h18","x19","att19","h19","x20","att20","h20"
        },("Incomplete teacher hooks",set(snap))
        # ModernBERT: h_att = hidden_states + attention_output
        a19=snap["x19"]+snap["att19"]
        a20=snap["x20"]+snap["att20"]
        assert torch.equal(snap["x19"],snap["h18"])
        assert torch.equal(snap["x20"],snap["h19"])
        result.append(dict(
            x=cpu_native(snap["h18"]),
            a19=cpu_native(a19),
            h19=cpu_native(snap["h19"]),
            a20=cpu_native(a20),
            h20=cpu_native(snap["h20"]),
            valid=cpu_native(b["attention_mask"].bool()),
        ))
        if (n+1)%12==0 or n+1==n_batches:
            print(label,"captured",n+1,"/",n_batches,
                  "precision",snap["h18"].dtype)
    return result

train_residual_cache=capture_residual_stages(
    train_items,TRAIN_CACHE_BATCHES,"TRAIN")
probe_residual_cache=capture_residual_stages(
    probe_items,PROBE_CACHE_BATCHES,"PROBE")
assert train_residual_cache and probe_residual_cache

# Important: explicitly verify that original teacher a19/y19, a20/y20
# are the actual two-residual computations and not arbitrary hook values.
@torch.inference_mode()
def validate_teacher_residual_algebra(row):
    r=cuda_grad_safe(row)
    with torch.autocast("cuda",dtype=amp_dtype):
        reconstructed19=r["a19"]+ORIGINAL_LAYERS[19].mlp(
            ORIGINAL_LAYERS[19].mlp_norm(r["a19"]))
        reconstructed20=r["a20"]+ORIGINAL_LAYERS[20].mlp(
            ORIGINAL_LAYERS[20].mlp_norm(r["a20"]))
    metrics={}
    for key,pred,target in (
        ("residual19",reconstructed19,r["h19"]),
        ("residual20",reconstructed20,r["h20"])):
        valid=r["valid"]
        delta=(pred.float()-target.float())[valid]
        rms=torch.sqrt((target.float()[valid]**2).mean().clamp_min(1e-8))
        rmse=torch.sqrt(delta.square().mean())/rms
        metrics[key]=dict(
            relative_rmse=float(rmse),
            max_abs=float(delta.abs().max()))
        assert float(rmse)<.0005,(key,metrics[key])
    return metrics

print("Frozen ModernBERT residual algebra:",
      validate_teacher_residual_algebra(probe_residual_cache[0]))
print("Captured four full-precision teacher stages; no final-state-only shortcuts.")


In [ ]:
#@title 7. Residual-faithful CeNN: frozen original norms, two residuals × two sliding steps
class SharedSpatialCeNN(nn.Module):
    def __init__(self,hidden,rank=96,long_range=False):
        super().__init__()
        self.down=nn.Linear(hidden,rank,bias=False)
        self.mix=nn.Linear(rank,rank,bias=False)
        self.local=nn.Conv1d(rank,rank,9,padding=128,dilation=32,
                             groups=rank,bias=False) # radius 128
        self.long_range=bool(long_range)
        if long_range:
            self.wide=nn.Conv1d(rank,rank,9,padding=512,dilation=128,
                                groups=rank,bias=False) # RF1025
            self.wide_gain=nn.Parameter(torch.tensor(-1.8))
        self.up=nn.Linear(rank,hidden,bias=False)
        nn.init.normal_(self.up.weight,std=.001)
    def forward(self,normalized,valid):
        m=valid.unsqueeze(-1).to(normalized.dtype)
        lower=F.silu(self.down(normalized*m))
        conv=self.local(lower.transpose(1,2).contiguous()).transpose(1,2)
        if self.long_range:
            broad=self.wide(lower.transpose(1,2).contiguous()).transpose(1,2)
            conv=conv+torch.sigmoid(self.wide_gain)*broad
        return self.up(F.silu(lower+self.mix(lower)+conv))*m

class LowRankResidualMLP(nn.Module):
    """Separate FFN approximation, NOT merged into CeNN attention residual."""
    def __init__(self,hidden,rank):
        super().__init__()
        self.down=nn.Linear(hidden,2*rank,bias=False)
        self.up=nn.Linear(rank,hidden,bias=False)
        nn.init.normal_(self.up.weight,std=.001)
    def forward(self,normalized,valid):
        a,b=self.down(normalized).chunk(2,dim=-1)
        return self.up(F.silu(a)*b)*valid[:,:,None].to(normalized.dtype)

class ResidualFaithfulTwoStep(nn.Module):
    """Mirror original two pre-LN ModernBERT residual pairs, in order.

    Original trained LayerNorms remain FROZEN; student attention-like cell
    weight is SHARED across steps, while each step has its own low-rank
    MLP residual, attentional gain and FFN gain.
    """
    def __init__(self,hidden,rank=96,ffn_rank=128,wide=False,tied=True):
        super().__init__()
        self.norm_att=nn.ModuleList([
            ORIGINAL_LAYERS[19].attn_norm,
            ORIGINAL_LAYERS[20].attn_norm
        ])
        self.norm_mlp=nn.ModuleList([
            ORIGINAL_LAYERS[19].mlp_norm,
            ORIGINAL_LAYERS[20].mlp_norm
        ])
        self.shared=bool(tied)
        self.att_cells=nn.ModuleList(
            [SharedSpatialCeNN(hidden,rank,wide)
             for _ in range(1 if tied else 2)]
        )
        self.ffn=nn.ModuleList([
            LowRankResidualMLP(hidden,ffn_rank),
            LowRankResidualMLP(hidden,ffn_rank)
        ])
        # Logits allow stable positive independent gains with genuine
        # residual identity. Initialize near no-op to avoid destroying
        # pretrained state before students learn.
        self.gain_att=nn.Parameter(torch.full((2,),-1.5))
        self.gain_ffn=nn.Parameter(torch.full((2,),-1.5))
        self.enable_attention=True
        self.enable_ffn=True
        self.ffn_backend="compressed"   # diagnostic option: native_teacher
        # All ORIGINAL pretrained norms are strictly frozen.
        for module in tuple(self.norm_att)+tuple(self.norm_mlp):
            module.requires_grad_(False)
    def attention_step(self,x,valid,index):
        assert index in (0,1)
        if not self.enable_attention:
            return x
        cell=self.att_cells[0 if self.shared else index]
        update=cell(self.norm_att[index](x),valid)
        return x+torch.sigmoid(self.gain_att[index])*update
    def ffn_step(self,a,valid,index):
        assert index in (0,1)
        if not self.enable_ffn:
            return a
        normalized=self.norm_mlp[index](a)
        if self.ffn_backend=="native_teacher":
            # Diagnostic control: retain the ORIGINAL learned GLU FFN,
            # but still replace native sliding attention with CeNN.
            # Frozen source modules are referenced, never cloned/optimized.
            update=ORIGINAL_LAYERS[19+index].mlp(normalized)
            return a+update
        assert self.ffn_backend=="compressed"
        update=self.ffn[index](normalized,valid)
        return a+torch.sigmoid(self.gain_ffn[index])*update
    def one_step(self,x,valid,index):
        # Exact original wiring: first attention residual, then FFN
        # residual with a FRESH normalization of the updated state.
        a=self.attention_step(x,valid,index)
        y=self.ffn_step(a,valid,index)
        return a,y
    def forward(self,x,valid,return_stages=False):
        # Prevent FP16 forward overflow inside the trainable adapter. The
        # frozen pretrained teacher remains under native mixed precision.
        if TRAIN_FP32_ADAPTER and x.device.type=="cuda":
            with torch.autocast("cuda",enabled=False):
                x=x.float()
                a19,h19=self.one_step(x,valid,0)
                a20,h20=self.one_step(h19,valid,1)
        else:
            a19,h19=self.one_step(x,valid,0)
            a20,h20=self.one_step(h19,valid,1)
        if return_stages:return dict(a19=a19,h19=h19,a20=a20,h20=h20)
        return h20

def normalized_rmse(pred,target,valid):
    m=valid[:,:,None].float()
    err=((pred.float()-target.float()).square()*m).sum()
    power=(target.float().square()*m).sum().clamp_min(1e-8)
    return torch.sqrt(err/power+1e-12)

def faithful_loss(student,row):
    mask=row["valid"]
    # All floating-point arithmetic in local reconstruction loss and
    # teacher-forced branches is FP32 (including outer autocast callers).
    with torch.autocast("cuda",enabled=False):
        return _faithful_loss_fp32(student,row)

def _faithful_loss_fp32(student,row):
    mask=row["valid"]
    rollout=student(row["x"].float(),mask,return_stages=True)
    # Two independent teacher-forced steps teach each update on its
    # *original* input before rollout compounds earlier errors.
    a19_forced=student.attention_step(row["x"].float(),mask,0)
    h19_forced=student.ffn_step(row["a19"].float(),mask,0)
    a20_forced=student.attention_step(row["h19"].float(),mask,1)
    h20_forced=student.ffn_step(row["a20"].float(),mask,1)
    rollout_rms=(
        .10*normalized_rmse(rollout["a19"],row["a19"],mask)+
        .20*normalized_rmse(rollout["h19"],row["h19"],mask)+
        .10*normalized_rmse(rollout["a20"],row["a20"],mask)+
        .30*normalized_rmse(rollout["h20"],row["h20"],mask))
    teacher_forced=(
        .08*normalized_rmse(a19_forced,row["a19"],mask)+
        .07*normalized_rmse(h19_forced,row["h19"],mask)+
        .07*normalized_rmse(a20_forced,row["a20"],mask)+
        .08*normalized_rmse(h20_forced,row["h20"],mask))
    # Fully supervise the original *two residual branch updates* per
    # layer, not only the final state after two sliding blocks.
    delta_loss=(
        .05*normalized_rmse(a19_forced-row["x"],
                            row["a19"]-row["x"],mask)+
        .05*normalized_rmse(h19_forced-row["a19"],
                            row["h19"]-row["a19"],mask)+
        .05*normalized_rmse(a20_forced-row["h19"],
                            row["a20"]-row["h19"],mask)+
        .05*normalized_rmse(h20_forced-row["a20"],
                            row["h20"]-row["a20"],mask))
    return rollout_rms+teacher_forced+delta_loss,rollout

@torch.inference_mode()
def stage_quality(student,cache):
    student.eval()
    stats={key:[] for key in ("a19","h19","a20","h20")}
    for original in cache:
        row=cuda_grad_safe(original)
        with torch.autocast("cuda",dtype=amp_dtype):
            pred=student(row["x"],row["valid"],return_stages=True)
        valid=row["valid"]
        for key in stats:
            student_valid=pred[key].float()[valid]
            teacher_valid=row[key].float()[valid]
            energy=(student_valid.square().sum()+
                    teacher_valid.square().sum()).clamp_min(1e-8)
            stats[key].append(dict(
                cosine=float(F.cosine_similarity(
                    student_valid,teacher_valid,dim=-1).mean()),
                rel_rmse=float(normalized_rmse(
                    pred[key],row[key],valid)),
                sym_error=float(
                    ((student_valid-teacher_valid).square().sum()/energy))
            ))
    avg={k:{m:float(np.mean([e[m] for e in v]))
            for m in ("cosine","rel_rmse","sym_error")}
         for k,v in stats.items()}
    return avg

class BypassEncoderBlock(nn.Module):
    def forward(self,hidden_states,*args,**kwargs):
        return (hidden_states,)

class GlobalWithFaithfulCeNN(nn.Module):
    def __init__(self,original,window,adapter):
        super().__init__()
        self.full=GlobalWindow18(original,window)
        self.adapter=adapter
    def forward(self,hidden_states,*args,**kwargs):
        h18=self.full(hidden_states,*args,**kwargs)[0]
        mask=kwargs["attention_mask"]
        valid=mask[:,0,0,:]
        valid=(valid.bool() if mask.dtype==torch.bool
               else valid.float()>-1e4)
        return (self.adapter(h18,valid),)

@contextmanager
def use_candidate(cell,width=1024):
    assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i]
               for i in range(28)),"Encoder already patched"
    try:
        teacher.encoder.layers[18]=GlobalWithFaithfulCeNN(
            ORIGINAL_LAYERS[18],width,cell)
        teacher.encoder.layers[19]=BypassEncoderBlock()
        teacher.encoder.layers[20]=BypassEncoderBlock()
        yield teacher
    finally:
        for i in (18,19,20):
            teacher.encoder.layers[i]=ORIGINAL_LAYERS[i]
        assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i]
                   for i in range(28)),"Failed to restore 28-layer teacher"

def trainable_params(cell):
    return [p for p in cell.parameters() if p.requires_grad]

def snapshot(cell):
    return {k:v.detach().cpu().clone()
            for k,v in cell.state_dict().items()}

def all_adapter_finite(cell):
    return all(torch.isfinite(p).all().item()
               for p in trainable_params(cell))

def grad_norm_and_finite(cell):
    params=[p for p in trainable_params(cell) if p.grad is not None]
    if not params:return 0.,False
    if not all(bool(torch.isfinite(p.grad).all()) for p in params):
        return float("inf"),False
    total=torch.linalg.vector_norm(torch.stack(
        [torch.linalg.vector_norm(p.grad.detach().float()) for p in params]))
    return float(total.detach()),True

hidden=int(teacher.encoder.config.hidden_size)
student=ResidualFaithfulTwoStep(
    hidden,CENN_RANK,FFN_RANK,
    USE_WIDE_BRANCH,TIE_LOCAL_WEIGHTS
).to(device).eval()

# Critical no-op residual preflight (all four additions must be identities).
one=cuda_grad_safe(probe_residual_cache[0])
student.enable_attention=False
student.enable_ffn=False
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    identity=student(one["x"],one["valid"],return_stages=True)
for key in ("a19","h19","a20","h20"):
    assert torch.equal(identity[key],one["x"]),(
        "Residual identity shortcut was broken",key)
student.enable_attention=True;student.enable_ffn=True

# Frozen original global SDPA and norms remain the same module objects.
assert student.norm_att[0] is ORIGINAL_LAYERS[19].attn_norm
assert student.norm_att[1] is ORIGINAL_LAYERS[20].attn_norm
assert student.norm_mlp[0] is ORIGINAL_LAYERS[19].mlp_norm
assert student.norm_mlp[1] is ORIGINAL_LAYERS[20].mlp_norm
assert not any(p.requires_grad for layer in ORIGINAL_LAYERS
               for p in layer.parameters())
assert len(trainable_params(student))>0

# Check gradient reaches both attention-like and BOTH MLP residual branches.
student.train()
row=cuda_grad_safe(train_residual_cache[0])
# Run the wiring/gradient preflight in FP32, not FP16, so
# a vanishing half-precision gradient cannot masquerade as broken wiring.
with torch.autocast("cuda",enabled=False):
    loss,_=faithful_loss(student,row)
assert torch.isfinite(loss)
loss.backward()
branch_grad={}
for key in ("att_cells.0.down.weight",
            "ffn.0.down.weight","ffn.1.down.weight",
            "gain_att","gain_ffn"):
    p=dict(student.named_parameters())[key]
    branch_grad[key]=float(p.grad.float().norm()) if p.grad is not None else 0.
    assert branch_grad[key]>0,("Missing trainable gradient",key,branch_grad)
student.zero_grad(set_to_none=True)
student.eval()
print("Residual identity, frozen pretrained norms, four stages, both FFN "
      "branch gradients and rollback preflight PASS.")
print("Gradient norms:",branch_grad)
print("Untrained four-stage metrics:",stage_quality(student,probe_residual_cache))


In [ ]:
#@title 8. Residual-faithful distillation with candidate-vs-no-update PROBE selection
# Do not reuse V4.0 residual-incompatible weights.
train_stage_history=[]
local_baseline=stage_quality(student,probe_residual_cache)
print("Before local training:",local_baseline)
# Absolute reference: original full18 + bypassed slides19/20.
student.enable_attention=False;student.enable_ffn=False
with use_candidate(student):
    probe_no_update=decision_score(probe_items,probe_teacher)
student.enable_attention=True;student.enable_ffn=True
with use_candidate(student):
    probe_initial=decision_score(probe_items,probe_teacher)
print("True no-residual-update PROBE:",probe_no_update)
print("Random student PROBE:",probe_initial)

def score_probe(metrics):
    """Lexicographic: preserve original decisions, gold and distribution."""
    return (metrics["teacher_agreement"],
            metrics["student_gold_accuracy"],
            -metrics["mean_js"])

# No-update is a valid fallback, with no extra trainable residual computation.
best_choice=dict(kind="no_update",quality=probe_no_update,
                 state=snapshot(student),stage=local_baseline,step=0)
def update_best(choice):
    global best_choice
    q=choice["quality"]
    # Explicit no-regression controls: a training checkpoint must NOT
    # improve teacher agreement while silently worsening gold accuracy
    # or becoming much less distribution-faithful than the no-op.
    qualifies=(
        q["teacher_agreement"]>=probe_no_update["teacher_agreement"] and
        q["student_gold_accuracy"]>=probe_no_update["student_gold_accuracy"] and
        q["mean_js"]<=probe_no_update["mean_js"]+.005
    )
    if qualifies and score_probe(q)>score_probe(best_choice["quality"]):
        best_choice=choice
        print("NEW best PROBE:",choice["kind"],"step",choice["step"],
              "agreement",choice["quality"]["teacher_agreement"],
              "gold",choice["quality"]["student_gold_accuracy"])
update_best(dict(kind="trained",quality=probe_initial,
                 state=snapshot(student),stage=local_baseline,step=0))

optimizer=torch.optim.AdamW(trainable_params(student),
                            lr=LEARNING_RATE,weight_decay=2e-4)
scaler=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
attempt=updates=0
while updates<LOCAL_TRAIN_STEPS:
    if attempt>LOCAL_TRAIN_STEPS+300:
        raise RuntimeError("Repeated FP16 overflow in V4.1; decrease LR.")
    row=cuda_grad_safe(train_residual_cache[attempt%len(train_residual_cache)])
    student.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        loss,_=faithful_loss(student,row)
    require_finite(loss,"V4.1 two residual pairs loss")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable_params(student),1.)
    before=scaler.get_scale()
    scaler.step(optimizer);scaler.update()
    attempt+=1
    if scaler.is_enabled() and scaler.get_scale()<before:continue
    updates+=1
    if updates==1 or updates%CHECK_EVERY==0 or updates==LOCAL_TRAIN_STEPS:
        student.eval()
        stages=stage_quality(student,probe_residual_cache)
        with use_candidate(student):
            q=decision_score(probe_items,probe_teacher)
        train_stage_history.append(dict(
            step=updates,loss=float(loss),stage=stages,task=q))
        update_best(dict(
            kind="trained",quality=q,state=snapshot(student),
            stage=stages,step=updates))
        print(f"V4.1 local step {updates:4}/{LOCAL_TRAIN_STEPS} "
              f"loss={float(loss):.5f} "
              f"att19 relRMSE={stages['a19']['rel_rmse']:.4f} "
              f"out19 relRMSE={stages['h19']['rel_rmse']:.4f} "
              f"att20 relRMSE={stages['a20']['rel_rmse']:.4f} "
              f"out20 relRMSE={stages['h20']['rel_rmse']:.4f} "
              f"PROBE agree={q['teacher_agreement']:.3f} "
              f"gold={q['student_gold_accuracy']:.3f}")

# Keep a genuinely trained starting state even if the no-update
# control wins local checkpoint selection. Task-level calibration must
# never silently backprop through disabled/identity branches.
last_trained_state=snapshot(student)
del optimizer,scaler
torch.cuda.empty_cache()
# Choose best supported checkpoint by PROBE, never by AUDIT.
student.load_state_dict(best_choice["state"],strict=True)
student.eval()
student.enable_attention=(best_choice["kind"]!="no_update")
student.enable_ffn=(best_choice["kind"]!="no_update")
print("Post-local checkpoint:",best_choice["kind"],best_choice["step"],
      "PROBE",best_choice["quality"])
print("No-update reference:",probe_no_update)
selected_local_file=OUTPUT_DIR/"v41_probe_selected_local.pt"
torch.save(dict(
    version="v41-residual-faithful-four-stage",
    original_model=SOURCE_MODEL,
    group=TARGET,window=1024,
    selected_kind=best_choice["kind"],
    checkpoint_quality=best_choice["quality"],
    original_norms_frozen=True,
    weights=snapshot(student),
    enable_attention=student.enable_attention,
    enable_ffn=student.enable_ffn,
    history=train_stage_history
),selected_local_file)
print("Resumable local candidate saved:",selected_local_file)


In [ ]:
#@title 9. Student-prefix task distillation + gold loss, PROBE-selected safety rollback
# When a trained adapter changes downstream input distributions, local
# teacher-forced MSE is insufficient. Calibrate through frozen 21..27
# using TRAIN-only task targets, and select by disjoint PROBE decisions.
task_history=[]
teacher_train_batches=[]

if RUN_TASK_CALIBRATION and TASK_CALIBRATION_STEPS>0:
    for bnum in range(TASK_TRAIN_BATCHES):
        selected=train_items[bnum*BATCH:(bnum+1)*BATCH]
        if len(selected)<BATCH:break
        b=direct_batch(selected)
        with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        truth=torch.softmax(logits.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),dim=-1)
        teacher_train_batches.append(dict(
            b={key:value.detach().cpu() for key,value in b.items()},
            probabilities=truth.detach().cpu(),
            gold=torch.tensor(
                [item["gold_index"] for item in selected],
                dtype=torch.long)
        ))
    assert teacher_train_batches,"No calibration TRAIN batches."
    # Activate the genuinely trained student; if it loses, the saved
    # PROBE optimum (even a zero-update baseline) remains available.
    start_weights=(
        last_trained_state if best_choice["kind"]=="no_update"
        else best_choice["state"])
    student.load_state_dict(start_weights,strict=True)
    student.enable_attention=True;student.enable_ffn=True
    student.train()
    opt=torch.optim.AdamW(trainable_params(student),
                          lr=TASK_LEARNING_RATE,weight_decay=1e-5)
    scaler=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
    attempts=updates=0
    with use_candidate(student):
        while updates<TASK_CALIBRATION_STEPS:
            if attempts>TASK_CALIBRATION_STEPS+300:
                raise RuntimeError("Task training repeatedly overflowed.")
            entry=teacher_train_batches[attempts%len(teacher_train_batches)]
            b={k:v.to(device).clone() for k,v in entry["b"].items()}
            p=entry["probabilities"].to(device).clone()
            gold=entry["gold"].to(device)
            student.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=amp_dtype):
                logits,_=teacher(**b) # student prefix + frozen later layers
                logp=F.log_softmax(logits.float().masked_fill(
                    ~b["marker_mask"].bool(),-1e9),dim=-1)
                distill=F.kl_div(logp,p,reduction="batchmean")
                supervised=F.nll_loss(logp,gold)
                # A small stage-reconstruction anchor keeps both norm/MLP
                # residual branches close to their original teacher targets.
                stage_row=cuda_grad_safe(
                    train_residual_cache[attempts%len(train_residual_cache)])
                local_loss,_=faithful_loss(student,stage_row)
                loss=distill+.10*supervised+.05*local_loss
            require_finite(loss,"V4.1 task KL + gold + stage residuals")
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params(student),.75)
            previous=scaler.get_scale()
            scaler.step(opt);scaler.update()
            attempts+=1
            if scaler.is_enabled() and scaler.get_scale()<previous:
                continue
            updates+=1
            if updates==1 or updates%20==0 or updates==TASK_CALIBRATION_STEPS:
                student.eval()
                # Already inside use_candidate, so do not nest wrapper.
                q=decision_score(probe_items,probe_teacher)
                stages=stage_quality(student,probe_residual_cache)
                task_history.append(dict(
                    step=updates,probe=q,
                    local=stages,loss=float(loss),
                    task_KL=float(distill)))
                update_best(dict(
                    kind="trained",quality=q,
                    state=snapshot(student),
                    stage=stages,step=LOCAL_TRAIN_STEPS+updates))
                print("Task step",updates,"/",
                      TASK_CALIBRATION_STEPS,"KL",round(float(distill),6),
                      "PROBE agreement",q["teacher_agreement"],
                      "gold",q["student_gold_accuracy"],
                      "local h20 rRMSE",round(stages["h20"]["rel_rmse"],5))
    del opt,scaler
    torch.cuda.empty_cache()
else:
    print("Task calibration disabled; keeping selected local checkpoint.")

# Revert to the PROBE optimum even if last training iteration degraded.
student.load_state_dict(best_choice["state"],strict=True)
student.enable_attention=(best_choice["kind"]!="no_update")
student.enable_ffn=(best_choice["kind"]!="no_update")
student.eval().requires_grad_(False)
with use_candidate(student):
    selected_probe=decision_score(probe_items,probe_teacher)
for key in ("teacher_agreement","student_gold_accuracy",
            "gold_accuracy_drop","mean_js"):
    assert np.isclose(
        selected_probe[key],best_choice["quality"][key],
        rtol=2e-4,atol=1e-6
    ),("PROBE quality changed after selected checkpoint restore",
       key,selected_probe[key],best_choice["quality"][key])
assert score_probe(selected_probe)>=score_probe(probe_no_update)
print("SELECTED V4.1:",best_choice["kind"],"step",best_choice["step"])
print("PROBE teacher agreement",selected_probe["teacher_agreement"],
      "gold accuracy",selected_probe["student_gold_accuracy"],
      "JS",selected_probe["mean_js"])
print("No-update PROBE control:",probe_no_update)
probe_selected_file=OUTPUT_DIR/"v41_probe_selected_residual_faithful.pt"
torch.save(dict(
    version="v41-trained-attention-and-MLP-residuals",
    checkpoint_kind=best_choice["kind"],
    stage=best_choice["stage"],
    selected_probe=selected_probe,
    no_update_probe=probe_no_update,
    adapter_state=snapshot(student),
    enabled=dict(attention=student.enable_attention,ffn=student.enable_ffn),
    training_history=train_stage_history,task_history=task_history,
    source_model=SOURCE_MODEL,trained_steps=best_choice["step"],
),probe_selected_file)
print("Selected checkpoint saved BEFORE audit:",probe_selected_file)


In [ ]:
#@title 10. Causal residual-branch checks on PROBE; ONE held-out AUDIT + paired GPU speed
# A/B ablations belong on PROBE: audit must not be used to select architecture.
probe_branch_ablations={}
old_att=student.enable_attention;old_ffn=student.enable_ffn
try:
    for name,att,ff in (
        ("no_student_updates",False,False),
        ("attention_residual_only",True,False),
        ("MLP_residual_only",False,True),
        ("both_original_order",True,True),
    ):
        student.enable_attention=att
        student.enable_ffn=ff
        with use_candidate(student):
            probe_branch_ablations[name]=decision_score(
                probe_items,probe_teacher)
        print("PROBE causal",name,
              probe_branch_ablations[name]["teacher_agreement"],
              probe_branch_ablations[name]["student_gold_accuracy"])
finally:
    student.enable_attention=old_att
    student.enable_ffn=old_ffn
with use_candidate(student):
    restored_probe=decision_score(probe_items,probe_teacher)
for key in ("teacher_agreement","student_gold_accuracy","mean_js"):
    assert np.isclose(restored_probe[key],selected_probe[key],
                      rtol=2e-4,atol=1e-6),(
        "Probe prediction changed after causal ablation toggles",key)

# AUDIT is evaluated exactly after selecting a checkpoint on PROBE.
audit_truth=teacher_probability_reference(audit_items,AUDIT_DECISIONS)
assert {r["case_id"] for r in audit_truth}.isdisjoint(
    {r["case_id"] for r in probe_teacher})
with use_candidate(student):
    heldout_audit=decision_score(audit_items,audit_truth)
print("SINGLE held-out audit result:",heldout_audit)

# Collect separate output/attention residual hidden-state targets on audit
# for DIAGNOSTICS ONLY, no checkpoint selection or retraining.
audit_stages=capture_residual_stages(
    audit_items,AUDIT_CACHE_BATCHES,"FINAL_AUDIT")
heldout_stage=stage_quality(student,audit_stages)
print("Held-out 4-stage residual diagnostics:",heldout_stage)

@torch.inference_mode()
def paired_full_cuda(b,repeats=BENCH_REPEATS):
    from contextlib import nullcontext
    durations={"native":[],"CeNN":[]}
    for name in durations:
        context=(use_candidate(student) if name=="CeNN"
                 else nullcontext())
        with context,torch.autocast("cuda",dtype=amp_dtype):
            for _ in range(3):teacher(**b)
    torch.cuda.synchronize()
    for repeat in range(repeats):
        sequence=(("native","CeNN","CeNN","native")
                  if repeat%2==0 else
                  ("CeNN","native","native","CeNN"))
        for name in sequence:
            context=(use_candidate(student) if name=="CeNN"
                     else nullcontext())
            with context,torch.autocast("cuda",dtype=amp_dtype):
                a=torch.cuda.Event(enable_timing=True)
                z=torch.cuda.Event(enable_timing=True)
                a.record();teacher(**b);z.record();z.synchronize()
                durations[name].append(float(a.elapsed_time(z)))
    old=float(np.median(durations["native"]))
    new=float(np.median(durations["CeNN"]))
    return dict(batch=int(b["input_ids"].shape[0]),
                length=int(b["input_ids"].shape[1]),
                original_ms=old,student_ms=new,
                speedup=old/max(new,1e-8),
                original_p95_ms=float(np.percentile(durations["native"],95)),
                student_p95_ms=float(np.percentile(durations["CeNN"],95)),
                trials=len(durations["native"]))

bench_batches=[batch(probe_items,0,1),
               batch(probe_items,0,BATCH),
               batch(probe_items,17,BATCH)]
paired_benchmark=[paired_full_cuda(b) for b in bench_batches]
for record in paired_benchmark:print("Paired full CUDA:",record)

# Measurement for the two original sliding blocks versus both residual-
# faithful CeNN updates, using precisely the same native h18 prefix.
@torch.inference_mode()
def segment_cuda(fn,repeats=BENCH_REPEATS):
    with torch.autocast("cuda",dtype=amp_dtype):
        for _ in range(3):fn()
        torch.cuda.synchronize()
        values=[]
        for _ in range(repeats):
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            a.record();fn();z.record();z.synchronize()
            values.append(float(a.elapsed_time(z)))
    return float(np.median(values))

# Capture original encoder 19/20 kwargs for valid segment comparison.
source_input=cuda_grad_safe(probe_residual_cache[0])
sample=batch(probe_items,0,BATCH)
original_kwargs={}
def catch_kwargs(mod,args,kwargs):
    original_kwargs.update(kwargs)
handle=ORIGINAL_LAYERS[19].register_forward_pre_hook(
    catch_kwargs,with_kwargs=True)
try:
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        teacher(**sample)
finally:
    handle.remove()
assert "sliding_window_mask" in original_kwargs
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    # The original 19 input should be the same teacher h18 captured above.
    original_two=lambda: ORIGINAL_LAYERS[20](
        ORIGINAL_LAYERS[19](source_input["x"],**original_kwargs)[0],
        **original_kwargs)[0]
    compressed_two=lambda: student(
        source_input["x"],source_input["valid"])
    original_local_ms=segment_cuda(original_two)
    student_local_ms=segment_cuda(compressed_two)
local_speedup=original_local_ms/max(student_local_ms,1e-8)
print("Native 19+20 versus faithful two-step CeNN:",
      original_local_ms,student_local_ms,local_speedup)

quality_pass=(
    heldout_audit["teacher_agreement"]>=.95 and
    heldout_audit["gold_accuracy_drop"]<=.01)
latency_pass=all(row["speedup"]>1.0 for row in paired_benchmark)
stage_pass=heldout_stage["h20"]["cosine"]>=.99
gates=dict(quality=bool(quality_pass),paired_speed=bool(latency_pass),
           stage_fidelity=bool(stage_pass))
gates["all_pass"]=all(gates.values())
print("V4.1 final acceptance gates:",gates)
print("Native checkpoint remains the deployment recommendation if not all_pass.")


In [ ]:
#@title 11. Export full architecture audit, corrected adapter and residual diagnostics
from datetime import datetime,timezone
import matplotlib.pyplot as plt

checkpoint=OUTPUT_DIR/"v41_residual_faithful_group18_20_EXPERIMENTAL.pt"
report_file=OUTPUT_DIR/"v41_residual_faithful_group18_20_report.json"
report=dict(
    version="4.1-residual-faithful-2attn-2ffn-CeNN",
    created_utc=datetime.now(timezone.utc).isoformat(),
    source_checkpoint=SOURCE_MODEL,
    source_model_weights_frozen=True,
    target_group=TARGET,
    full_attention_window=1024,
    original_sliding_radius=64,
    local_CeNN_radius=128,
    long_CeNN_rf=1025 if USE_WIDE_BRANCH else None,
    local_convolution_shared_across_sliding_steps=TIE_LOCAL_WEIGHTS,
    original_norms_reused_and_frozen=True,
    original_residual_topology="h_att = x + attention(LN_att(x)); "
                               "h_out = h_att + FFN(LN_mlp(h_att)), twice",
    teacher_stages=["a19","h19","a20","h20"],
    original_residual_check=validate_teacher_residual_algebra(
        probe_residual_cache[0]),
    gradients_on_both_residual_branches=branch_grad,
    original_probe=baseline,
    true_no_update_probe=probe_no_update,
    random_initial_probe=probe_initial,
    selected_probe=selected_probe,
    selected_state_kind=best_choice["kind"],
    selected_training_step=best_choice["step"],
    selected_probe_stage=best_choice["stage"],
    stage_training=train_stage_history,
    task_training=task_history,
    causal_probe_ablations=probe_branch_ablations,
    heldout_audit=heldout_audit,
    heldout_teacher_stage=heldout_stage,
    paired_cuda=paired_benchmark,
    native_sliding_pair_ms=original_local_ms,
    new_CeNN_pair_ms=student_local_ms,
    pair_speedup=local_speedup,
    gates=gates,
    source_train_cases=TRAIN_CASES,
    source_dev_cases=DEV_CASES,
    official_test_used=False,
    checkpoint_status="experimental" if not gates["all_pass"]
                      else "provisional_pass_requires_external_test",
    note="V4.0 trained CeNN had worse audit than disabling the local "
         "updates. V4.1 preserves both attention and FFN residuals per "
         "sliding layer; uses exact original LayerNorm modules; selects "
         "best via PROBE, only then evaluates AUDIT.",
)
report_file.write_text(json.dumps(report,indent=2,default=str),encoding="utf-8")
torch.save(dict(
    version=report["version"],source_model=SOURCE_MODEL,
    target_group=TARGET,
    config=dict(rank=CENN_RANK,ffn_rank=FFN_RANK,
                full_window=1024,wide=USE_WIDE_BRANCH,
                tied_local=TIE_LOCAL_WEIGHTS),
    checkpoint_kind=best_choice["kind"],
    enable_attention=student.enable_attention,
    enable_ffn=student.enable_ffn,
    adapter_state=snapshot(student),
    selected_probe=selected_probe,
    audit=heldout_audit,gates=gates,
    deployable=bool(gates["all_pass"])
),checkpoint)
print("Report",report_file)
print("Adapter checkpoint",checkpoint)
print("Selected checkpoint",best_choice["kind"],"PROBE",selected_probe)
print("Heldout AUDIT",heldout_audit)
print("Paired CUDA speedups",
      [round(row["speedup"],3) for row in paired_benchmark])
print("Pass all requirements:",gates["all_pass"])

keys=["a19","h19","a20","h20"]
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(keys,[heldout_stage[k]["rel_rmse"] for k in keys])
axes[0].set(ylabel="Relative RMSE",title="Four original residual stages")
axes[1].bar(["Original","Student"],
            [float(np.median([x["original_ms"] for x in paired_benchmark])),
             float(np.median([x["student_ms"] for x in paired_benchmark]))])
axes[1].set(ylabel="CUDA latency (ms)",title="Paired full-model inference")
fig.tight_layout()
figure_path=OUTPUT_DIR/"v41_residual_faithful_diagnostics.png"
fig.savefig(figure_path,dpi=150,bbox_inches="tight")
plt.show()
print("Diagnostics",figure_path)
